<a href="https://colab.research.google.com/github/ilaydacepniogluu-sys/AkademiQ_DataScience/blob/main/AkademiQHAFTA8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#SVM ( Support Vector Machine)

#Parametreler


Margin =


C = Gerçekte her zaman gruplar kusursuz ayrılmaz. SVM'nin hata toleransına karşılık gelen değerdir.

C büyükse "hiç hata istemiyorum" demiş olursunuz.

C küçükse "Birkaç hata olabilir." demiş olursunuz.


C = 0.001-0.1 arası: çok esnek. Model hata yapmaktan çekilmez.

C = 1 başlangıç noktasıdır. Varsayılan değer

C = 10-100 arası: Katı model. Model hata yapsın istemez.

C = 1000 ve üzeri: Tehlikeli bölge. Model eğitim verisini ezberler.

C'nin kaç belirleneceğini GridSearchCV ile seçebiliriz.



Kernel trick = Düzlemde ayrılamayan veriler. Veri iki boyutlu düzlemde ayırılamıyorsa kernel fonksiyonlarıyla veriyi daha yüksek bir boyuta taşıyıp,orada ayrılır.


Kernel yüksek boyuttaki kordinatları hesaplamaz noktalar arası benzerlik hesaplar. Bu hesaplama kısayoluna kernel trick denir.


KERNEL TÜRLERİ

RBF (Radial Basis Function)


Gamma = RBF kernel'deki balonların büyüklüğünü ayarlar. Gamma yüksekse balonlar küçük,model yerel karar verir. Gamma düşükse balonlar büyük. Model daha geniş bölgelere bakarak karar verir.


SVM nerelerde kullanılır?

*Pataloji laboratuvarı ( Kanserli hücre mi? sağlıklı hücre mi? ayrımında)

*Finans tarafında (dolandırıcılık var mı?)

*NASA


NOT: çok az veriyle çalışmak zorunda kaldığı, gerçek zamanlı karar verdiği bir yapıda SVM ciddi bir potansiyel taşıyan algoritmadır.


!!! Mülakatlarda çok sık karşılaşılır.

In [6]:

import pandas as pd
import numpy as np
import pickle, os #kendi dosya yapılarınızla çalışabileceğiniz, pythonda yazılan algortimayı paketlemeyi sağlar.
#Python'da oluşturulan model veya nesneleri dosyaya kaydetmek ve daha sonra tekrar yüklemek için kullanılır.(pickle)
#os: Dosya ve klasörlerle çalışmak; dosya yollarını kontrol etmek, oluşturmak ve yönetmek için kullanılır.
from sklearn.model_selection import train_test_split, GridSearchCV
#train_test_split: Veriyi eğitim (train) ve test olarak ayırmak için kullanılır.
#GridSearchCV: Makine öğrenmesi modelinde kullanılacak en uygun parametreleri denemek ve bulmak için kullanılır.
from sklearn.preprocessing import StandardScaler, LabelEncoder
#StandardScaler: Sayısal verileri standartlaştırmak/ölçeklendirmek için kullanılır. Verilerin farklı ölçeklerde olmasından kaynaklanan sorunları azaltır.
#LabelEncoder: Kategorik/metinsel değerleri sayısal değerlere dönüştürmek için kullanılır.
from sklearn.svm import SVC
#SVC (Support Vector Classifier): Sınıflandırma yapmak için kullanılan bir makine öğrenmesi algoritmasıdır.
from sklearn.ensemble import RandomForestClassifier
#RandomForestClassifier: Birden fazla karar ağacını birlikte kullanarak sınıflandırma yapan makine öğrenmesi algoritmasıdır.
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix)

#Bunların hepsi modelinin ne kadar başarılı olduğunu ölçmek için kullanılır.
#accuracy_score : Modelin yaptığı tahminlerin yüzde kaçının doğru olduğunu gösterir.
#precision_score: Modelin pozitif tahminlerinin ne kadarının gerçekten pozitif olduğunu gösterir.
#recall_score: Gerçekte pozitif olanların ne kadarını doğru yakaladığını gösterir.
#f1_score : Precision ve Recall değerlerini birlikte değerlendirir. Özellikle sınıfların dengesiz olduğu durumlarda kullanışlıdır.
#roc_auc_score : Modelin sınıfları birbirinden ayırt etme başarısını ölçer. Genellikle değer 1'e yaklaştıkça modelin ayırt etme performansı daha yüksektir.
#confusion_matrix : Modelin yaptığı doğru ve yanlış tahminleri tablo halinde gösterir.



In [7]:
df =pd.read_csv("musteri_rfm_dt.csv", sep=";" ) # Veri setimiz noktalı virgül ile ayrıldığı için tablo formatında görüntülemek için bir separator eklemek gerekir.
#Separator: ayırıcı demektir. Python'da özellikle print() ve split() gibi fonksiyonlarda karşına çıkar. print içindeki değerlerin arasına ne konulacağını belirler.
print(f"Satır: {df.shape[0]} | Sütun: {df.shape[1]}")
print(df.head())

Satır: 2371 | Sütun: 11
  musteri_id  recency_gun  frequency  ...      sehir  segment_mevcut  kayip_riski
0  MUS-10418           30          8  ...   İstanbul           sadık            0
1  MUS-11976           18         14  ...  Gaziantep        şampiyon            0
2  MUS-12327          127          5  ...   İstanbul           sadık            0
3  MUS-11241           55          7  ...    Antalya           sadık            0
4  MUS-10536           36          3  ...     Ankara          riskli            1

[5 rows x 11 columns]


In [10]:
#EDA ( keşifsel analiz )

for v, cnt in df['kayip_riski'].value_counts().items():
  print(f'{v}  {cnt}  ({cnt/len(df)*100:.1f} % )')


  for col in ['recency_gun', 'frequency', 'monetary']:
    print(f"{col:12s} -> min: {df[col].min():6.0f} max:{df[col].max():6.0f} ort:{df[col].mean():7.1f}")


0  1200  (50.6 % )
recency_gun  -> min:      0 max:   709 ort:  100.2
frequency    -> min:      1 max:    18 ort:    5.1
monetary     -> min:      7 max:110871 ort:12575.3
1  1171  (49.4 % )
recency_gun  -> min:      0 max:   709 ort:  100.2
frequency    -> min:      1 max:    18 ort:    5.1
monetary     -> min:      7 max:110871 ort:12575.3


In [11]:
df['kanal_enc'] = LabelEncoder().fit_transform(df['dominant_kanal'])
df['sehir_enc'] = LabelEncoder().fit_transform(df['sehir'])


features = ['recency_gun', 'frequency', 'monetary', 'ort_sepet', 'kanal_enc', 'sehir_enc']
X = df[features]
y = df['kayip_riski']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Eğitim:  {len(X_train)}  satır | test :  {len(X_test)} satır')

Eğitim:  1896  satır | test :  475 satır


In [ ]:
#NOT: TÜM VERİ SETİ AYNI ÖLÇEKTE OLMAZSA SVM ÇALIŞMAZ.
#Standart Scaler şart. (SVM için zorunlu)


In [12]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print("Ölçekleme Tamamlandı.")
print(f'monetary önce ort: {X_train['monetary'].mean():.1f} -> sonraki ort: {X_train_sc[:,2].mean():.2f} ')

Ölçekleme Tamamlandı.
monetary önce ort: 12629.9 -> sonraki ort: 0.00 


In [14]:
#SVM Eğitimi

svm = SVC(kernel = 'rbf', C= 1, gamma = 'scale', probability =True, random_state=42 )
svm.fit(X_train_sc, y_train)

y_pred = svm.predict(X_test_sc)
y_prob = svm.predict_proba(X_test_sc)[:,1]

print(f'Accuracy: {accuracy_score(y_test,y_pred):.4f}')
print(f'precision: {precision_score(y_test,y_pred):.4f}')
print(f'recall: {recall_score(y_test,y_pred):.4f}')
print(f'f1 Score: {f1_score(y_test,y_pred):.4f}')
print(f'roc_auc: {roc_auc_score(y_test,y_prob):.4f}')


Accuracy: 0.9663
precision: 0.9363
recall: 1.0000
f1 Score: 0.9671
roc_auc: 0.9903


NOT: SVM den alınan sonuç her zaman ya 0 ya da 1 dir.